# Population & infrastructure risk exposure

**Question:** how many people and which facilities are exposed to a given hazard *before* any event?

**Method** (after INFORM / the MapAction–IFRC children-exposure work):
1. **Hazard mask** — compare the hazard raster to the intensity threshold → 1 exposed / 0 not.
2. **Overlay population** — resample the mask to the 1 km population grid and multiply. Everyone is counted first;
   children under 5 and under 15 are then broken out of the same grid.
3. **Zonal statistics** — sum exposed population per subdivision for every return period, then combine them
   into an **annual average** (each return period's extra exposure weighted by 1 / return period).
4. **Facilities** — read the hazard value at each facility's centroid and record the lowest return period that exposes it.

In [ ]:
# --- Reserved parameters (injected by the Notebook Factory; defaults keep the notebook runnable standalone) ---
run_id = ""
inputs_dir = ""
output_dir = "outputs"
area = {}                      # {id, code, name, level, country, country_name, bbox}
area_geojson = ""              # FeatureCollection with the target area
subdivisions_geojson = ""      # FeatureCollection with the aggregation subdivisions
stac_item = {}                 # the run's STAC item (dict); unused by this template
stac_item_json = ""
stac_collection = ""
stac_api_url = ""              # root of the item's catalog
stac_catalog_profile = ""      # "generic" | "montandon"
filters = {}                   # the run's filter tags, e.g. {"hazard": ["flood"], "use_case": ["uc1"]}
hazard = ""

# --- User parameters ---
hazard = "flood"
intensity_threshold = None      # None = the hazard's default (flood 0.5, cyclone 33, earthquake 0.22)
return_period = 100             # years; earthquakes only have 475
climate_scenario = "current"    # current | future (ignored for earthquakes)
aggregation_level = 2
infrastructure_layers = ["health"]   # health | schools | airports | ports
population_year = 2025          # WorldPop R2025A covers 2015-2030 (later years are projections)
datasets_url = ""               # hazard rasters: a folder or URL (https://, s3://, az://) holding the hazard .tif files

In [ ]:
import json, os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

warnings.filterwarnings("ignore", message=".*geographic CRS.*")
Path(output_dir).mkdir(parents=True, exist_ok=True)

def load_area_and_subdivisions():
    """Read the boundaries prepared by the app. Fails loudly rather than inventing an area."""
    if not (area_geojson and Path(area_geojson).exists()):
        raise FileNotFoundError(f"area_geojson not found: {area_geojson!r}")
    area_gdf = gpd.read_file(area_geojson)
    has_subs = subdivisions_geojson and Path(subdivisions_geojson).exists()
    return area_gdf, (gpd.read_file(subdivisions_geojson) if has_subs else area_gdf.copy())

area_gdf, subdivisions = load_area_and_subdivisions()
area_name = area.get("name") or area_gdf.iloc[0].get("name", "Area")
iso = str(area.get("country") or area_gdf.iloc[0]["country"]).upper()
print(f"Area: {area_name} ({iso}) · {len(subdivisions)} subdivisions")
subdivisions[["code", "name", "level"]].head()

## 1 · Datasets

* **Hazard rasters** are global GeoTIFFs (ideally Cloud-Optimized) directly inside the folder or URL given as the `datasets_url`
  parameter (`https://`, `s3://`, `az://`). Only the part inside the area is read.
* **Population** is [WorldPop](https://www.worldpop.org) R2025A: people per 1 km cell for the year `population_year`, with national totals
  aligned to the UN World Population Prospects. It is read through the [WorldPop API](https://api.worldpop.org/v3), which sends only the
  cells inside the area (a large area is requested in tiles). **Children** are WorldPop's age classes 0–4 (under 5) and 0–14 (under 15).
  R2025A is published as an alpha release. The `WORLDPOP_API_KEY` secret raises the API's limits. © WorldPop, University of Southampton,
  CC BY 4.0: population doi:10.5258/SOTON/WP00839, age and sex doi:10.5258/SOTON/WP00841.
* **Health facilities, schools, airports and ports** are OpenStreetMap features from the per-country GeoParquet extracts that the
  [Humanitarian OpenStreetMap Team](https://www.hotosm.org) (HOT) publishes on HDX each month, read straight from the file (no local copy).
  Every feature becomes one point, its centroid. Health is `healthcare=*` or hospital, clinic, doctors, dentist, pharmacy; schools are
  `amenity=school`; airports `aeroway=aerodrome`; ports `industrial=port`, `harbour=yes` or `landuse=port|harbour`.

In [ ]:
import hashlib
import tempfile
import time
import uuid
import urllib.request
import duckdb
import rasterio
import requests
import shapely
from rasterio.mask import mask as rio_mask
from rasterio.merge import merge as rio_merge
from rasterio.warp import Resampling, reproject
from rasterstats import zonal_stats

# Where the hazard rasters live: the `datasets_url` parameter, a folder or a URL (https://, s3://, az://) that holds
# the hazard .tif files directly. The files are too big to ship in a template bundle, so the template points at them.
# A private bucket's credentials must be template secrets, never part of this value.
DATASETS_DIR = datasets_url.strip().rstrip("/")
if not DATASETS_DIR:
    raise ValueError("Set the `datasets_url` parameter to the folder or URL that holds the hazard rasters")
DATASETS_REMOTE = "://" in DATASETS_DIR
if not DATASETS_REMOTE and not Path(DATASETS_DIR).is_dir():
    raise FileNotFoundError(f"datasets_url is not a folder: {DATASETS_DIR!r}")
# Scratch space for the population rasters we fetch. Safe to delete: anything here can be fetched again.
# TMPDIR decides where that is, so a deployment can point it at a scratch volume.
CACHE_DIR = Path(tempfile.gettempdir()) / "notebook-factory-cache"
# Remote rasters: don't list the bucket folder on every open, and only treat .tif files as rasters.
os.environ.setdefault("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR")
os.environ.setdefault("CPL_VSIL_CURL_ALLOWED_EXTENSIONS", ".tif")

# (hazard, climate scenario) -> file pattern, return periods (years), default threshold.
# File names inside `datasets_url`; `{rp}` is the return period.
RPS = [10, 50, 100, 500, 1000]
HAZARDS = {
    ("flood", "current"): ("wrl_risk_ras_wri_flood-rp{rp}y.tif", RPS, 0.5),
    ("flood", "future"): ("wrl_risk_ras_wri_flood-future-rp{rp}y.tif", RPS, 0.5),
    ("cyclone", "current"): ("wrl_risk_ras_4tu_cyclone-rp{rp}y.tif", RPS, 33.0),
    ("cyclone", "future"): ("wrl_risk_ras_4tu_cyclone-future-rp{rp}y.tif", RPS, 33.0),
    ("earthquake", "current"): ("wrl_risk_ras_gem_earthquake-rp{rp}y.tif", [475], 0.22),
}

# Cloud-hosted datasets: WorldPop (through its API) and OpenStreetMap (HOT's GeoParquet extracts, one file per country and theme).
WORLDPOP_API = "https://api.worldpop.org/v3"
WORLDPOP_KEY = os.environ.get("WORLDPOP_API_KEY", "")   # optional: a key raises the API's area and request limits
WORLDPOP_NODATA = -99999.0
USER_AGENT = {"User-Agent": "notebook-factory (IFRC Montandon use case 1)"}
HDX_PACKAGE = "https://data.humdata.org/api/3/action/package_show?id=hotosm_{iso}_{theme}"
HOT_URL = "https://production-raw-data-api.s3.amazonaws.com/ISO3/{ISO}/_layers/osm/{theme}.parquet"
HEALTH_AMENITIES = ("hospital", "clinic", "doctors", "dentist", "pharmacy")  # older tagging; healthcare=* is the newer one

def _in(values):
    return ", ".join(f"'{v}'" for v in values)

# infrastructure layer -> (HOT theme = file name, SQL filter on its columns; the columns are OpenStreetMap tags)
OSM_LAYERS = {
    "health": ("health_facilities", f"healthcare IS NOT NULL OR amenity IN ({_in(HEALTH_AMENITIES)})"),
    "schools": ("education_facilities", "amenity = 'school'"),
    "airports": ("airports", "aeroway = 'aerodrome'"),   # the airport itself, not its runways, taxiways or helipads
    "ports": ("sea_ports", "industrial = 'port' OR harbour = 'yes' OR landuse IN ('port', 'harbour')"),
}
OSM_SOURCES = {}   # layer -> URL of the HOT file that was read
LAYER_LABELS = {"health": "Health facilities", "schools": "Schools", "airports": "Airports", "ports": "Ports"}
LAYER_COLORS = {"health": [31, 36, 48], "schools": [30, 120, 200], "airports": [120, 60, 170], "ports": [0, 140, 130]}  # map colour of each layer

# Population groups: key -> (WorldPop layer, age range). Everyone is the headline; children are broken out of the same grid.
POP_GROUPS = {"": ("population", None), "under5": ("agesex", (0, 4)), "under15": ("agesex", (0, 14))}

def get_dataset(pattern, **fmt):
    """Location (path or URL) of a hazard file inside `datasets_url`."""
    rel = pattern.format(**fmt)
    if DATASETS_REMOTE:
        return f"{DATASETS_DIR}/{rel}"  # existence is checked when the raster is opened
    path = Path(DATASETS_DIR) / rel
    if not path.exists():
        raise FileNotFoundError(f"dataset not found: {path}")
    return str(path)

def open_raster(path):
    """rasterio.open with the location in the error (remote failures are otherwise hard to read)."""
    try:
        return rasterio.open(path)
    except rasterio.errors.RasterioIOError as e:
        raise RuntimeError(f"Cannot open raster {path}: {e}") from e

class WorldPopAreaTooBig(Exception):
    """The API refused the area as too large or too complex. `limit_km2` is its area limit for this caller."""
    def __init__(self, message, limit_km2=None):
        super().__init__(message)
        self.limit_km2 = limit_km2

def worldpop_raster_request(geom, year, layer, age_range):
    """One POST /raster call: GeoTIFF bytes of people per 1 km cell (cells whose centre is inside `geom`)."""
    body = {"geojson": json.loads(shapely.to_geojson(geom)), "year": int(year), "layer": layer, "resolution": "1km"}
    if layer == "agesex":
        body["bands"] = "sum"   # one band: the people of the chosen ages, both sexes
        body["age_range"] = list(age_range)
    headers = {**USER_AGENT, **({"X-API-Key": WORLDPOP_KEY} if WORLDPOP_KEY else {})}
    for attempt in range(5):
        r = requests.post(f"{WORLDPOP_API}/raster", json=body, headers=headers, timeout=(30, 300))
        if r.status_code in (429, 502, 503, 504) and attempt < 4:
            error = r.json().get("error") if "json" in r.headers.get("content-type", "") else ""
            if error == "rate_limit_exceeded":
                break   # the day's quota is used up: waiting a few seconds won't help
            time.sleep(min(float(r.headers.get("Retry-After", 3 * (attempt + 1))), 60))
            continue
        break
    if r.ok:
        return r.content
    try:
        err = r.json()
    except ValueError:
        err = {}
    if r.status_code == 413 and err.get("error") in ("area_limit_exceeded", "too_many_parts", "geometry_too_complex", "raster_too_large"):
        raise WorldPopAreaTooBig(err.get("message", ""), (err.get("details") or {}).get("limit_km2"))
    raise RuntimeError(f"WorldPop API {r.status_code} {err.get('error', '')}: {err.get('message', r.text[:200])}")

def _polygons(geom):
    return shapely.MultiPolygon([p for p in shapely.get_parts(geom) if p.geom_type == "Polygon"])

def _lean(geom, max_vertices=40_000):
    """Simplify until the API's 50,000-vertex limit is safe: a few hundred metres of edge are far below a 1 km cell."""
    tolerance = 0.0005
    while shapely.get_num_coordinates(geom) > max_vertices:
        geom = shapely.simplify(geom, tolerance)
        tolerance *= 2
    return geom

def split_into_tiles(geom, n):
    """Cut `geom` along an n x n grid of its bounding box (empty pieces are dropped)."""
    xmin, ymin, xmax, ymax = geom.bounds
    xs, ys = np.linspace(xmin, xmax, n + 1), np.linspace(ymin, ymax, n + 1)
    tiles = []
    for i in range(n):
        for j in range(n):
            piece = geom.intersection(shapely.box(xs[i], ys[j], xs[i + 1], ys[j + 1]))
            if not piece.is_empty:
                piece = _polygons(piece)
                if not piece.is_empty:
                    tiles.append(_lean(piece))
    return tiles

def fetch_worldpop(geom, year, layer, age_range=None):
    """Path of a GeoTIFF of WorldPop people per 1 km cell covering `geom` (EPSG:4326, nodata -99999), cached.
    An area over the API's limit is cut into a grid of tiles that are requested one by one and merged
    (every tile is on WorldPop's 1 km grid, so they join exactly)."""
    geom = _lean(_polygons(geom))
    key = hashlib.sha1(f"{shapely.to_wkb(geom).hex()}|{year}|{layer}|{age_range}".encode()).hexdigest()[:16]
    path = CACHE_DIR / "worldpop" / f"{layer}_{year}_{key}.tif"
    if path.exists():
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    n = 1
    while True:
        tiles = [geom] if n == 1 else split_into_tiles(geom, n)
        try:
            blobs = [worldpop_raster_request(t, year, layer, age_range) for t in tiles]
            break
        except WorldPopAreaTooBig as e:
            if n >= 8:
                raise RuntimeError(f"The area is too large for the WorldPop API even in {n * n} tiles ({e}). Choose a smaller area.") from e
            if n == 1 and e.limit_km2:
                area_km2 = gpd.GeoSeries([geom], crs=4326).to_crs("ESRI:54009").area.iloc[0] / 1e6
                n = max(2, int(np.ceil(np.sqrt(area_km2 / (0.8 * e.limit_km2)))))
            else:
                n += 1
            print(f"WorldPop API: {e} Requesting the area in {n * n} tiles.")
    files, srcs = [rasterio.MemoryFile(b) for b in blobs], []
    try:
        srcs = [f.open() for f in files]
        data, transform = (srcs[0].read(), srcs[0].transform) if len(srcs) == 1 else rio_merge(srcs, nodata=WORLDPOP_NODATA)
        tmp = path.with_suffix(f".{uuid.uuid4().hex}.part")   # unique per run: runs can overlap
        with rasterio.open(tmp, "w", driver="GTiff", height=data.shape[1], width=data.shape[2], count=data.shape[0], dtype="float32",
                           crs="EPSG:4326", transform=transform, nodata=WORLDPOP_NODATA, compress="deflate") as dst:
            dst.write(data.astype("float32"))
        tmp.replace(path)   # only a complete file ever gets the final name
    finally:
        for src_ in srcs:
            src_.close()
        for f in files:
            f.close()
    people = float(np.where(data == WORLDPOP_NODATA, 0.0, data).sum())
    print(f"WorldPop {layer}{age_range or ''} {year}: {len(tiles)} request(s), {people:,.0f} people in the area's cells")
    return path

def hot_layer_url(iso, theme):
    """URL of HOT's GeoParquet for one country and theme. HDX lists it as that dataset's GeoParquet resource, so ask HDX first
    (the path is HOT's to change) and fall back to the usual path."""
    usual = HOT_URL.format(ISO=iso.upper(), theme=theme)
    try:
        r = requests.get(HDX_PACKAGE.format(iso=iso.lower(), theme=theme), headers=USER_AGENT, timeout=30)
        r.raise_for_status()
        for resource in r.json()["result"]["resources"]:
            if (resource.get("format") or "").lower() == "geoparquet":
                return resource["url"]
    except (requests.RequestException, KeyError, ValueError):
        pass
    return usual

def hot_last_modified(url):
    """When HOT last rebuilt this file (HTTP Last-Modified), which is the date of its OpenStreetMap snapshot."""
    try:
        req = urllib.request.Request(url, method="HEAD", headers=USER_AGENT)
        with urllib.request.urlopen(req, timeout=30) as r:
            return r.headers.get("Last-Modified", "unknown")
    except Exception:
        return "unknown"

def load_osm_points(iso, bbox, layer):
    """OpenStreetMap features of one infrastructure layer inside bbox (xmin, ymin, xmax, ymax), each reduced to its centroid.
    HOT cuts one extract per country and theme (the columns are the OSM tags); DuckDB reads only the rows in the box."""
    theme, where = OSM_LAYERS[layer]
    url = OSM_SOURCES[layer] = hot_layer_url(iso, theme)
    xmin, ymin, xmax, ymax = bbox
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs; INSTALL spatial; LOAD spatial;")
    try:
        df = con.execute(f"""
            SELECT id AS osm_id, ST_X(ST_Centroid(geom)) AS x, ST_Y(ST_Centroid(geom)) AS y
            FROM read_parquet('{url}')
            WHERE bbox.xmin <= {xmax} AND bbox.xmax >= {xmin} AND bbox.ymin <= {ymax} AND bbox.ymax >= {ymin}
              AND ({where})
        """).df()
    except duckdb.Error as e:
        raise RuntimeError(f"No usable HOT OpenStreetMap '{theme}' layer for {iso} ({url}): {e}") from e
    return gpd.GeoDataFrame(df[["osm_id"]], geometry=gpd.points_from_xy(df.x, df.y), crs=4326)

def read_window(path, geoms):
    """Read only the part of a raster inside `geoms`; pixels outside (or nodata) become 0."""
    with open_raster(path) as src:
        data, transform = rio_mask(src, geoms, crop=True, filled=True)
        nodata, crs = src.nodata, src.crs
    arr = data[0].astype("float64")
    invalid = ~np.isfinite(arr)
    if nodata is not None:
        invalid |= arr == nodata
    arr[invalid] = 0.0
    return arr, transform, crs

def match_grid(arr, transform, crs, shape, like_transform, like_crs):
    """Resample `arr` onto another raster's grid (nearest neighbour keeps hazard values intact)."""
    out = np.zeros(shape, dtype="float64")
    reproject(source=arr, destination=out, src_transform=transform, src_crs=crs,
              dst_transform=like_transform, dst_crs=like_crs, resampling=Resampling.nearest)
    return out

def zonal_sum(arr, transform, zones):
    """Sum raster values inside each zone polygon (pixel centres)."""
    stats = zonal_stats(list(zones.geometry), arr, affine=transform, stats="sum", nodata=-999)
    return np.array([s["sum"] or 0.0 for s in stats])

def sample_raster(path, points):
    """Raster value at each point (0 for nodata / outside the raster)."""
    with open_raster(path) as src:
        vals = np.array([v[0] for v in src.sample([(p.x, p.y) for p in points])], dtype="float64")
        nodata = src.nodata
    vals[~np.isfinite(vals)] = 0.0
    if nodata is not None:
        vals[vals == nodata] = 0.0
    return vals

# Resolve the parameters. No data for the requested combination is an error, never fake data.
scenario = "current" if hazard == "earthquake" else climate_scenario
if (hazard, scenario) not in HAZARDS:
    raise ValueError(f"No dataset for hazard={hazard!r}, climate_scenario={scenario!r} yet")
pattern, return_periods, default_threshold = HAZARDS[(hazard, scenario)]
threshold = default_threshold if intensity_threshold is None else intensity_threshold
chosen_rp = return_periods[0] if len(return_periods) == 1 else return_period
if chosen_rp not in return_periods:
    raise ValueError(f"return_period must be one of {return_periods}, got {return_period}")
print(f"{hazard} · {scenario} · threshold {threshold} · {chosen_rp}-yr (all of {return_periods} are computed)")

## 2 · Population exposure

For every return period: mask the hazard where it exceeds the threshold, multiply by population and sum per subdivision, first for
everyone and then for children under 5 and under 15 (the same mask on the same grid). The **annual average** adds each return period's
extra exposure divided by that return period.

In [ ]:
area_geom = shapely.union_all(list(area_gdf.geometry))
geoms = list(area_gdf.geometry)

# People per 1 km cell: everyone, then children under 5 and under 15 (all on the same grid).
pops, pop_tf, pop_crs = {}, None, None
for group, (wp_layer, ages) in POP_GROUPS.items():
    arr, tf, crs = read_window(fetch_worldpop(area_geom, population_year, wp_layer, ages), geoms)
    if pop_tf is None:
        pop_tf, pop_crs = tf, crs
    elif arr.shape != pops[""].shape or tf != pop_tf:
        raise RuntimeError(f"WorldPop grid for {group!r} does not match the total population grid")
    pops[group] = arr

def col(base, group):
    """Column of a statistic for a population group: the total keeps the plain name (exposed), a group adds its key (exposed_under5)."""
    return f"{base}_{group}" if group else base

stats = subdivisions.copy()
for group, pop in pops.items():
    stats[col("population", group)] = zonal_sum(pop, pop_tf, stats)

annual, prev = {g: np.zeros(len(stats)) for g in pops}, {g: None for g in pops}
for rp in sorted(return_periods):
    haz, haz_tf, haz_crs = read_window(get_dataset(pattern, rp=rp), geoms)
    hit = match_grid(haz, haz_tf, haz_crs, pops[""].shape, pop_tf, pop_crs) > threshold
    for group, pop in pops.items():
        exposed_rp = zonal_sum(pop * hit, pop_tf, stats)
        stats[f"{col('exposed', group)}_rp{rp}"] = exposed_rp
        # the first return period counts in full; later ones only add the people newly exposed
        annual[group] += exposed_rp / rp if prev[group] is None else (exposed_rp - prev[group]) / rp
        prev[group] = exposed_rp

for group in pops:
    pop_col, exp_col = col("population", group), col("exposed", group)
    stats[exp_col] = stats[f"{exp_col}_rp{chosen_rp}"]
    stats[f"{exp_col}_share"] = (stats[exp_col] / stats[pop_col].replace(0, np.nan)).fillna(0)
    stats[f"{exp_col}_annual_avg_pct"] = np.where(stats[pop_col] > 0, annual[group] / stats[pop_col] * 100, 0.0)
print(f"Population: {stats['population'].sum():,.0f} · exposed at {chosen_rp}-yr: {stats['exposed'].sum():,.0f}")
print(f"Children exposed at {chosen_rp}-yr: under 5 {stats['exposed_under5'].sum():,.0f} · under 15 {stats['exposed_under15'].sum():,.0f}")
if not stats["exposed_under5"].sum() <= stats["exposed_under15"].sum() <= stats["exposed"].sum() * 1.001:
    print("Warning: the age groups are not nested (under 5 <= under 15 <= everyone); check the WorldPop rasters")

## 3 · Facilities

Health facilities, schools, airports and ports come from OpenStreetMap (HOT's per-country extracts); each is one point, its centroid. Each facility is assigned to a subdivision, then the hazard raster is read at its location for every return
period. `lowest_rp` is the most frequent (lowest) return period that exposes it; empty means not exposed.

In [ ]:
fac_frames = []
for layer in infrastructure_layers:
    if layer not in OSM_LAYERS:
        raise ValueError(f"No dataset for infrastructure layer {layer!r} yet")
    fac_id = "osm_id"
    fac = load_osm_points(iso, tuple(area_gdf.total_bounds), layer)
    fac = gpd.sjoin(fac[[fac_id, "geometry"]], stats[["code", "geometry"]], how="inner").drop(columns="index_right")
    fac = fac.drop_duplicates(subset=fac_id).reset_index(drop=True)
    fac["layer"] = layer
    fac["lowest_rp"] = np.nan
    for rp in sorted(return_periods):
        hit = sample_raster(get_dataset(pattern, rp=rp), fac.geometry) > threshold
        fac[f"exposed_rp{rp}"] = hit
        fac.loc[hit & fac["lowest_rp"].isna(), "lowest_rp"] = rp
    fac["exposed"] = fac[f"exposed_rp{chosen_rp}"]
    fac_frames.append(fac)
facilities = gpd.GeoDataFrame(pd.concat(fac_frames, ignore_index=True), crs=4326) if fac_frames else gpd.GeoDataFrame(
    columns=["layer", "code", "exposed", "geometry"], crs=4326)

for layer in infrastructure_layers:
    f = facilities[facilities.layer == layer]
    stats[f"{layer}_total"] = f.groupby("code").size().reindex(stats["code"]).fillna(0).astype(int).values
    stats[f"{layer}_exposed"] = f[f.exposed].groupby("code").size().reindex(stats["code"]).fillna(0).astype(int).values
    print(f"{layer}: {len(f):,} facilities · exposed at {chosen_rp}-yr: {int(f.exposed.sum()):,}")
if infrastructure_layers:
    print("Facilities: OpenStreetMap via HOT's per-country extracts, files updated "
          + ", ".join(f"{layer} {hot_last_modified(OSM_SOURCES[layer])}" for layer in infrastructure_layers))

In [ ]:
cols = ["name", "population", "exposed", "exposed_share", "exposed_annual_avg_pct",
        "population_under5", "exposed_under5", "exposed_under5_share", "population_under15", "exposed_under15", "exposed_under15_share"]
cols += [c for c in stats.columns if c.endswith("_exposed") or c.endswith("_total")]
table = stats[cols].sort_values("exposed", ascending=False)
for c in ("population", "exposed", "population_under5", "exposed_under5", "population_under15", "exposed_under15"):
    table[c] = table[c].round(0)
table["exposed_annual_avg_pct"] = table["exposed_annual_avg_pct"].round(2)
table.to_csv(Path(output_dir) / "exposure_by_subdivision.csv", index=False)
stats.to_file(Path(output_dir) / "exposure_by_subdivision.geojson", driver="GeoJSON")
facilities.drop(columns="geometry").to_csv(Path(output_dir) / "facilities_exposure.csv", index=False)
fmt = {c: "{:,.0f}" for c in cols if c.startswith(("population", "exposed")) and not c.endswith(("share", "pct"))}
fmt.update({"exposed_share": "{:.1%}", "exposed_under5_share": "{:.1%}", "exposed_under15_share": "{:.1%}"})
table.style.format(fmt).background_gradient(subset=["exposed_share"], cmap="OrRd")

## Results

In [ ]:
# Visualisation: lonboard for maps, manywidgets for charts, controls and stats.
# All of these are anywidgets, so they stay interactive in the published (kernel-free) page.
from matplotlib import colormaps
from lonboard import Map, PolygonLayer, ScatterplotLayer
from manywidgets import Chart, Column, Legend, Row, Slider, Stat
from manywidgets.lonboard import LayerToggle

def colors_for(values, cmap="OrRd", vmin=None, vmax=None):
    v = np.asarray(values, dtype=float)
    vmin = np.nanmin(v) if vmin is None else vmin
    vmax = np.nanmax(v) if vmax is None else vmax
    norm = (v - vmin) / (vmax - vmin) if vmax > vmin else np.zeros_like(v)
    rgba = colormaps[cmap](np.nan_to_num(norm))
    return (rgba * 255).astype(np.uint8)

def legend_for(cmap, vmin, vmax, title="", n=5, fmt="{:.0%}"):
    stops = np.linspace(vmin, vmax, n)
    rgb = colors_for(stops, cmap, vmin, vmax)[:, :3]
    return Legend(entries=[[[int(c) for c in rgb[i]], fmt.format(stops[i])] for i in range(n)], title=title)

def bar_chart(names, values, title, y_label, color="#E4572E"):
    chart = Chart(chart_type="bar", title=title, y_label=y_label, height=max(280, 22 * len(names) + 120), legend_enabled=False)
    chart.set_options(scales={"x": {"type": "category", "ticks": {"autoSkip": False, "maxRotation": 60, "minRotation": 30}}})
    chart.add_series(x=[str(n) for n in names], y=[float(v) for v in values], name=y_label, series_type="bar", color=color)
    return chart

def fmt_int(v):
    return f"{float(v):,.0f}"

child_stats = [Stat(label="Under 5 exposed", value=fmt_int(stats["exposed_under5"].sum())),
               Stat(label="Under 15 exposed", value=fmt_int(stats["exposed_under15"].sum()))]
fac_stats = [Stat(label=f"{LAYER_LABELS[l]} exposed", value=int(stats[f"{l}_exposed"].sum())) for l in infrastructure_layers]
Row(
    Stat(label="Population", value=fmt_int(stats["population"].sum())),
    Stat(label="Exposed population", value=fmt_int(stats["exposed"].sum())),
    Stat(label="Share exposed", value=f"{stats['exposed'].sum() / max(stats['population'].sum(), 1):.1%}"),
    *child_stats,
    *fac_stats,
    gap="16px",
)

### Map — exposed population share per subdivision

In [ ]:
vmax = max(float(stats["exposed_share"].max()), 0.01)
zones_layer = PolygonLayer.from_geopandas(stats[["name", "population", "exposed", "exposed_share", "geometry"]], get_fill_color=colors_for(stats["exposed_share"], "OrRd", 0, vmax), get_line_color=[60, 60, 60], line_width_min_pixels=1, opacity=0.75)
layers = [zones_layer]
fac_layers = {}
for layer in infrastructure_layers:
    fx = facilities[(facilities.layer == layer) & facilities.exposed]
    if len(fx):
        fac_layers[layer] = ScatterplotLayer.from_geopandas(fx[["layer", "geometry"]], get_fill_color=LAYER_COLORS[layer], radius_min_pixels=3)
        layers.append(fac_layers[layer])
Map(layers)

In [ ]:
toggles = [LayerToggle(layer=zones_layer, label="Exposed share")] + [LayerToggle(layer=l, label=f"Exposed {LAYER_LABELS[k].lower()}") for k, l in fac_layers.items()]
Row(Column(*toggles, gap="6px"), legend_for("OrRd", 0, vmax, title="Exposed population share"), gap="24px", align="start")

### Exposed population by subdivision

In [ ]:
bar_chart(table["name"], table["exposed"], title=f"{area_name} · {hazard} ≥ {threshold} · {chosen_rp}-yr · {scenario}", y_label="Exposed population")

## Reproduce or adapt

Download this notebook, set `datasets_url` to your hazard data (a folder or URL), set `area_geojson` / `subdivisions_geojson` to your own
boundaries and re-run it with standard Jupyter tooling. The published page is a frozen artefact; the method is not.